# LifeGuard AI — Module 3: Health Check (EDA, SMOTE + Calibration & Error Analysis)

> **Disclaimer**: Awareness and educational screening only — **NOT medical advice or a clinical diagnosis**.

This notebook audits the Pima Indians Diabetes (`768` rows) and UCI Cleveland Heart Disease (`303` rows) cohorts, verifies leakage-free preprocessing + SMOTE inside `imblearn.pipeline.Pipeline`, evaluates `CalibratedClassifierCV` (ROC-AUC, Recall, Brier score, calibration curves), and inspects untouched 20% test-set errors.

In [ ]:
import json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("../../..").resolve()
META_PATH = REPO_ROOT / "ml" / "health" / "artifacts" / "metadata.json"

entries = json.loads(META_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(entries)} Health Check model reports.")

## 1. Clinical Data Audit & Leakage Prevention

- **Diabetes (`diabetes.csv`)**: Biological zeros (`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`) are converted to `NaN` inside `DiabetesClinicalCleaner` and median-imputed with missingness indicators inside CV folds.
- **Heart Disease (`heart_cleveland.csv`)**: Categorical codes (`cp`, `restecg`, `slope`, `thal`) are one-hot encoded and missing `ca`/`thal` values are imputed inside `ColumnTransformer`.
- **SMOTE + Probability Calibration**: `SMOTE` runs strictly on training folds inside `imblearn.pipeline.Pipeline`. Because oversampling shifts base priors, `CalibratedClassifierCV(method="sigmoid", cv=5)` calibrates probabilities against untouched validation folds.

In [ ]:
for entry in entries:
    print(f"=== {entry['sub_model'].upper()} TRACK ===")
    print("Dataset:", entry["dataset"])
    print("Splits :", entry["splits"])
    print("Thresholds:", entry["thresholds"])
    display(pd.DataFrame(entry["comparison_table"]))

## 2. Calibration Reliability, Untouched Test Metrics & Error Analysis

In [ ]:
for entry in entries:
    print(f"=== {entry['sub_model'].upper()} TEST PERFORMANCE ({entry['model_name']}) ===")
    print("At 0.50 :", entry["metrics"]["test_at_0_50"])
    print("At tuned:", entry["metrics"]["test_at_tuned_elevated"])
    print("Alert recall (moderate + elevated):", entry["metrics"]["test_alert_recall_including_moderate"])
    print("Calibration curve (test bins):", entry["metrics"]["calibration_curve_test"])
    print("Confusion matrix @ tuned elevated:", entry["error_analysis"]["confusion_matrix_at_t_high"])
    print("Limitations:")
    for lim in entry["limitations"]:
        print(" -", lim)
    print()